In [80]:
import numpy as np
import pandas as pd
import scanpy as sc
import h5py
import anndata as ad
from pathlib import Path

In [81]:
# read preprocessed log1p transformed data with 1000 most variable genes and selected genes from the paper
data_dir = '../data/frangieh'
adata = sc.read_h5ad(f"{data_dir}/dataset_hv_tim.h5ad")

### Compute log2FC

In [82]:
adata_ctl = adata[adata.obs['perturbation_2'].eq('Control')].copy()
adata_IFN = adata[adata.obs['perturbation_2'].eq('Co-culture')].copy()
adata_co = adata[adata.obs['perturbation_2'].eq('IFNγ')].copy()

In [87]:
def get_rank_genes_groups(adata, file_prefix):
    file = Path(data_dir) / f'{file_prefix}_ranked_tim.h5ad'
    if file.is_file():
        print(f"Loading ranked genes from {file.absolute()}")
        data = sc.read_h5ad(file)
        return data
    else:
        print(f"Computing ranked genes and saving to {file.absolute()}")
        data = adata.copy()
        sc.tl.rank_genes_groups(data, 'perturbation', reference='control', method='wilcoxon')
        data.write(file)
        return data

In [88]:
adata_ctl = get_rank_genes_groups(adata_ctl, 'adata_ctl')
adata_IFN = get_rank_genes_groups(adata_IFN, 'adata_IFN')
adata_co = get_rank_genes_groups(adata_co, 'adata_co')

Loading ranked genes from /tmp/pycharm_project_c95b1eb4/notebooks/../data/frangieh/adata_ctl_ranked_tim.h5ad
Loading ranked genes from /tmp/pycharm_project_c95b1eb4/notebooks/../data/frangieh/adata_IFN_ranked_tim.h5ad
Loading ranked genes from /tmp/pycharm_project_c95b1eb4/notebooks/../data/frangieh/adata_co_ranked_tim.h5ad


In [89]:
# df with cols groups:perturbation, names:gene, logfoldchange:log2FC perturbation vs. control
adata_ctl_FC = sc.get.rank_genes_groups_df(adata_ctl, None)
adata_IFN_FC = sc.get.rank_genes_groups_df(adata_IFN, None)
adata_co_FC = sc.get.rank_genes_groups_df(adata_co, None)